# McNemar's Test — Rëndësia Statistikore e Dallimeve mes Modeleve

Teston nëse dallimet në accuracy mes çift-modeleve (Baseline SVM, DistilBERT, XLM-RoBERTa) 
janë statistikisht domethënëse, apo mund të jenë thjesht rastësi/variancë e ndarjes së të dhënave.

## 1. Ngarkimi dhe bashkimi i parashikimeve

In [2]:
import pandas as pd

svm_df = pd.read_csv('../../results/predictions_svm_test.csv')
distilbert_df = pd.read_csv('../../results/predictions_distilbert_test.csv')
xlmr_df = pd.read_csv('../../results/predictions_xlmr_test.csv')

print("SVM:", svm_df.shape)
print("DistilBERT:", distilbert_df.shape)
print("XLM-R:", xlmr_df.shape)

print("\nSVM head:\n", svm_df.head())
print("\nDistilBERT head:\n", distilbert_df.head())

SVM: (16175, 2)
DistilBERT: (16175, 2)
XLM-R: (16175, 2)

SVM head:
       y_true y_pred_svm
0    Ekonomi    Ekonomi
1       Botë       Botë
2   Politikë   Politikë
3  Lifestyle  Lifestyle
4    Kulturë    Kulturë

DistilBERT head:
       y_true y_pred_distilbert
0    Ekonomi           Ekonomi
1       Botë              Botë
2   Politikë          Politikë
3  Lifestyle         Lifestyle
4    Kulturë           Kulturë


In [3]:
print("A përputhen y_true mes SVM dhe DistilBERT?", (svm_df['y_true'].values == distilbert_df['y_true'].values).all())
print("A përputhen y_true mes SVM dhe XLM-R?", (svm_df['y_true'].values == xlmr_df['y_true'].values).all())

A përputhen y_true mes SVM dhe DistilBERT? True
A përputhen y_true mes SVM dhe XLM-R? True


In [4]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


## 2. Përgatitja e tabelave 2×2 dhe llogaritja e McNemar's Test

In [5]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

def bej_mcnemar(y_true, y_pred_a, y_pred_b, emri_a, emri_b):
    """
    Ndërton tabelën 2x2 dhe kryen McNemar's test mes dy modeleve.
    """
    korrekt_a = (y_true == y_pred_a)
    korrekt_b = (y_true == y_pred_b)
    
    # n11: të dy modelet e qëllojnë saktë
    # n10: vetëm A e qëllon saktë (B gabon)
    # n01: vetëm B e qëllon saktë (A gabon)
    # n00: të dy gabojnë
    n11 = np.sum(korrekt_a & korrekt_b)
    n10 = np.sum(korrekt_a & ~korrekt_b)
    n01 = np.sum(~korrekt_a & korrekt_b)
    n00 = np.sum(~korrekt_a & ~korrekt_b)
    
    tabela = [[n11, n10], [n01, n00]]
    
    print(f"\n{'='*50}")
    print(f"McNemar's Test: {emri_a} vs {emri_b}")
    print(f"{'='*50}")
    print(f"Të dy korrekt (n11): {n11}")
    print(f"Vetëm {emri_a} korrekt (n10): {n10}")
    print(f"Vetëm {emri_b} korrekt (n01): {n01}")
    print(f"Të dy gabim (n00): {n00}")
    
    rezultat = mcnemar(tabela, exact=False, correction=True)
    print(f"\nStatistika chi-square: {rezultat.statistic:.4f}")
    print(f"p-value: {rezultat.pvalue:.6f}")
    
    if rezultat.pvalue < 0.05:
        print(f"→ Dallimi ËSHTË statistikisht domethënës (p < 0.05)")
    else:
        print(f"→ Dallimi S'ËSHTË statistikisht domethënës (p >= 0.05)")
    
    return rezultat

y_true = svm_df['y_true'].values
y_pred_svm = svm_df['y_pred_svm'].values
y_pred_distilbert = distilbert_df['y_pred_distilbert'].values
y_pred_xlmr = xlmr_df['y_pred_xlmr'].values

# Tre çiftet e krahasimit
res_svm_vs_distilbert = bej_mcnemar(y_true, y_pred_svm, y_pred_distilbert, "SVM (Baseline)", "DistilBERT")
res_distilbert_vs_xlmr = bej_mcnemar(y_true, y_pred_distilbert, y_pred_xlmr, "DistilBERT", "XLM-RoBERTa")
res_svm_vs_xlmr = bej_mcnemar(y_true, y_pred_svm, y_pred_xlmr, "SVM (Baseline)", "XLM-RoBERTa")


McNemar's Test: SVM (Baseline) vs DistilBERT
Të dy korrekt (n11): 12718
Vetëm SVM (Baseline) korrekt (n10): 836
Vetëm DistilBERT korrekt (n01): 1174
Të dy gabim (n00): 1447

Statistika chi-square: 56.5020
p-value: 0.000000
→ Dallimi ËSHTË statistikisht domethënës (p < 0.05)

McNemar's Test: DistilBERT vs XLM-RoBERTa
Të dy korrekt (n11): 13397
Vetëm DistilBERT korrekt (n10): 495
Vetëm XLM-RoBERTa korrekt (n01): 782
Të dy gabim (n00): 1501

Statistika chi-square: 64.0532
p-value: 0.000000
→ Dallimi ËSHTË statistikisht domethënës (p < 0.05)

McNemar's Test: SVM (Baseline) vs XLM-RoBERTa
Të dy korrekt (n11): 12902
Vetëm SVM (Baseline) korrekt (n10): 652
Vetëm XLM-RoBERTa korrekt (n01): 1277
Të dy gabim (n00): 1344

Statistika chi-square: 201.8538
p-value: 0.000000
→ Dallimi ËSHTË statistikisht domethënës (p < 0.05)


# Ruaj rezultatet

In [6]:
import json

mcnemar_rezultate = {
    'SVM_vs_DistilBERT': {
        'n11': 12718, 'n10': 836, 'n01': 1174, 'n00': 1447,
        'chi_square': 56.5020, 'p_value': 0.000000, 'statistikisht_domethenes': True
    },
    'DistilBERT_vs_XLMR': {
        'n11': 13397, 'n10': 495, 'n01': 782, 'n00': 1501,
        'chi_square': 64.0532, 'p_value': 0.000000, 'statistikisht_domethenes': True
    },
    'SVM_vs_XLMR': {
        'n11': 12902, 'n10': 652, 'n01': 1277, 'n00': 1344,
        'chi_square': 201.8538, 'p_value': 0.000000, 'statistikisht_domethenes': True
    }
}

RESULTS_PATH = r'C:\Users\denis\Downloads\Desktop\diploma-project\results'
with open(f'{RESULTS_PATH}\\mcnemar_rezultate.json', 'w') as f:
    json.dump(mcnemar_rezultate, f, indent=2)

print("Rezultatet e McNemar's Test u ruajtën.")

Rezultatet e McNemar's Test u ruajtën.


## 3. Analiza e gjeneralizueshmërisë sipas portalit

In [7]:
test_df = pd.read_csv('../../data/processed/test.csv')

# Verifikim: a përputhet rendi me predictions?
print("Përputhje y_true:", (test_df['label'].values == svm_df['y_true'].values).all())

# Bashko source-in me parashikimet e secilit model
test_df['pred_svm'] = svm_df['y_pred_svm'].values
test_df['pred_distilbert'] = distilbert_df['y_pred_distilbert'].values
test_df['pred_xlmr'] = xlmr_df['y_pred_xlmr'].values

# Accuracy per portal, per model
portal_stats = []
for portal, group in test_df.groupby('source'):
    if len(group) < 30:  # anashkalo portale me shumë pak shembuj në test set
        continue
    acc_svm = (group['label'] == group['pred_svm']).mean()
    acc_distilbert = (group['label'] == group['pred_distilbert']).mean()
    acc_xlmr = (group['label'] == group['pred_xlmr']).mean()
    portal_stats.append({
        'portal': portal, 'n_artikuj': len(group),
        'acc_svm': acc_svm, 'acc_distilbert': acc_distilbert, 'acc_xlmr': acc_xlmr
    })

portal_df = pd.DataFrame(portal_stats).sort_values('n_artikuj', ascending=False)
print(portal_df.to_string(index=False))

Përputhje y_true: True
       portal  n_artikuj  acc_svm  acc_distilbert  acc_xlmr
    Telegrafi       2149 0.874360        0.899023  0.905537
    Insajderi       2110 0.799052        0.810900  0.824171
   GazetaBlic       2038 0.793425        0.802748  0.833660
     Ballkani       2018 0.873142        0.892963  0.918731
   Klankosova       1947 0.817668        0.863893  0.872625
 Indeksonline       1578 0.862484        0.889734  0.907478
     Lajme_al       1360 0.839706        0.876471  0.874265
GazetaExpress        972 0.865226        0.873457  0.901235
      Bota_al        828 0.861111        0.865942  0.888889
   VizionPlus        537 0.970205        0.972067  0.979516
    Lajmi_net        512 0.666016        0.660156  0.738281
  JavaNews_al         80 0.712500        0.787500  0.775000
   Kallxo.com         46 0.913043        0.956522  0.934783


## 4. Shembuj konkretë të artikujve të keq-klasifikuar (XLM-RoBERTa)

In [8]:
# Fokusohemi te konfuzioni kryesor: Kuriozitete/Lifestyle/Shëndet
kategori_problematike = ['Kuriozitete', 'Lifestyle', 'Shëndet']

gabime = test_df[
    (test_df['label'] != test_df['pred_xlmr']) &
    (test_df['label'].isin(kategori_problematike)) &
    (test_df['pred_xlmr'].isin(kategori_problematike))
].copy()

gabime['snippet'] = gabime['text_final'].str[:300]

# Merr 5 shembuj (mund t'i shohësh të gjithë dhe të zgjedhësh më ilustruesit)
shembuj = gabime.sample(5, random_state=42)[['label', 'pred_xlmr', 'source', 'snippet']]
for idx, row in shembuj.iterrows():
    print(f"\n{'='*70}")
    print(f"E vërtetë: {row['label']} | Parashikuar: {row['pred_xlmr']} | Portal: {row['source']}")
    print(f"Teksti: {row['snippet']}...")


E vërtetë: Kuriozitete | Parashikuar: Shëndet | Portal: GazetaBlic
Teksti: Këto dy mënyra do t’ju zgjojnë në mëngjes pa ndihmën e kafes. Nëse nuk je adhuruese e kafes, është e vështirë të ndihesh e zgjuar pa goditjen që të jep kafeina, për të cilën shumica e njerëzve kanë nevojë. Nëse nuk e konsideron veten njeri mëngjesi, zgjimi çdo ditë mund të jetë i vështirë, ndërsa të...

E vërtetë: Shëndet | Parashikuar: Lifestyle | Portal: Klankosova
Teksti: Cilët persona duhet kenë kujdes nga majdanozi?. Kujdes! Majdanozi mund të ketë disa të meta ndaj dhe apelohet që të mos konsumohet për disa kategori personash. Ndër ta janë gratë shtatzëna, pasi mund të rrisë rrezikun për abort, bëhet fjalë veçanërisht për gjethet e tij, pasi kërkohet se në rast kon...

E vërtetë: Lifestyle | Parashikuar: Kuriozitete | Portal: Lajmi_net
Teksti: Si ta dalloni një njeri të keq në vetëm 5 minuta. Këshilla Në jetën e përditshme shpesh nuk mund ta imagjinojmë se kush na gënjen, apo lëndon. Ata që shpërndajnë gën